<a href="https://colab.research.google.com/github/beruscoder/architectures/blob/main/vit_from_scratch_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torchvision
import matplotlib.pyplot as plt
import torchvision.transforms as transforms
from torch.utils.data import dataloader
import torch.nn as nn
import torch.optim as optim

In [2]:
transformation = transforms.Compose([transforms.ToTensor()])

In [3]:
train_dataset = torchvision.datasets.MNIST(root ='./data', train = True, download=True, transform=transformation)
val_dataset = torchvision.datasets.MNIST(root ='./data', train = False, download=True, transform=transformation)

100%|██████████| 9.91M/9.91M [00:02<00:00, 4.93MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 131kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.25MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 9.28MB/s]


In [12]:
#hyperparameter
#variables
batch_size = 64
img_size = 28
patch_size = 7
num_channels = 1
num_patches = (img_size // patch_size) ** 2
num_heads = 1
embed_dim = 16
mlp_dim = 16
transformer_units = 1

In [5]:
#create train and val batches
train_data = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True)
val_data = torch.utils.data.DataLoader(valset, batch_size=batch_size,
                                          shuffle=False)

In [13]:
class PatchEmbedding(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embed = nn.Conv2d(num_channels, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        x = self.patch_embed(x)
        x = x.flatten(2)
        x = x.transpose(1,2)
        return x

In [7]:
images, labels = next(iter(train_data))
patch_embed = nn.Conv2d(num_channels, embed_dim, kernel_size = patch_size , stride =patch_size)
embedded_image = patch_embed(images)
print(images.shape)
print(patch_embed(images).shape)
print(embedded_image.flatten(2).shape)

torch.Size([64, 1, 28, 28])
torch.Size([64, 20, 4, 4])
torch.Size([64, 20, 16])


In [14]:
class TransformerArchitecture(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm_1 = nn.LayerNorm(embed_dim)
        self.self_attention = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)
        self.layer_norm_2 = nn.LayerNorm(embed_dim)
        self.multi_layer_perceptron = nn.Sequential(
            nn.Linear(embed_dim, mlp_dim),
            nn.GELU(),
            nn.Linear(mlp_dim, embed_dim)
        )

    def forward(self, x):
        residual_1 = x
        attention_output = self.self_attention(self.layer_norm_1(x),self.layer_norm_1(x),self.layer_norm_1(x))[0]
        x = attention_output + residual_1
        residual_2 = x
        mlp_output = self.multi_layer_perceptron(self.layer_norm_2(x))
        x = mlp_output + residual_2
        return x


In [15]:
class VisionTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.patch_embedding = PatchEmbedding()
        self.cls_token = nn.Parameter(torch.randn(1,1,embed_dim))
        self.pos_embed = nn.Parameter(torch.randn(1, (img_size // patch_size) ** 2 + 1, embed_dim))
        self.transformer_layers = nn.Sequential(*[TransformerArchitecture() for _ in range(transformer_units)])

        self.mlp_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 10)
        )

    def forward(self,x):
        x = self.patch_embedding(x)
        B = x.size(0)

        cls_tokens = self.cls_token.expand(B , -1, -1)
        x = torch.cat((cls_tokens, x), dim=1)
        x = x + self.pos_embed
        x = self.transformer_layers(x)
        x = x[:,0]
        x = self.mlp_head(x)
        return x

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = VisionTransformer().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

In [17]:
for epoch in range(5):
    model.train()
    total_loss = 0
    correct_epoch = 0
    total_epoch = 0
    print(f"\nEpoch {epoch+1}")

    for batch_idx, (images, labels) in enumerate(train_data):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)

        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss+=loss.item()
        preds = outputs.argmax(dim=1)

        correct = (preds == labels).sum().item()
        accuracy = 100.0 * correct / labels.size(0)

        correct_epoch += correct
        total_epoch += labels.size(0)

        if batch_idx % 100 == 0:
            print(f"  Batch {batch_idx+1:3d}: Loss = {loss.item():.4f}, Accuracy = {accuracy:.2f}%")

    epoch_acc = 100.0 * correct_epoch / total_epoch
    print(f"==> Epoch {epoch+1} Summary: Total Loss = {total_loss:.4f}, Accuracy = {epoch_acc:.2f}%")


Epoch 1
  Batch   1: Loss = 2.5029, Accuracy = 4.69%
  Batch 101: Loss = 1.5156, Accuracy = 48.44%
  Batch 201: Loss = 1.0090, Accuracy = 65.62%
  Batch 301: Loss = 0.7662, Accuracy = 76.56%
  Batch 401: Loss = 0.8836, Accuracy = 68.75%
  Batch 501: Loss = 0.9445, Accuracy = 70.31%
  Batch 601: Loss = 0.8426, Accuracy = 68.75%
  Batch 701: Loss = 0.5552, Accuracy = 82.81%
  Batch 801: Loss = 0.6378, Accuracy = 75.00%
  Batch 901: Loss = 0.6890, Accuracy = 75.00%
==> Epoch 1 Summary: Total Loss = 816.4910, Accuracy = 70.37%

Epoch 2
  Batch   1: Loss = 0.5693, Accuracy = 82.81%
  Batch 101: Loss = 0.3590, Accuracy = 84.38%
  Batch 201: Loss = 0.5399, Accuracy = 85.94%
  Batch 301: Loss = 0.3632, Accuracy = 87.50%
  Batch 401: Loss = 0.7174, Accuracy = 75.00%
  Batch 501: Loss = 0.8543, Accuracy = 68.75%
  Batch 601: Loss = 0.6797, Accuracy = 79.69%
  Batch 701: Loss = 0.4824, Accuracy = 84.38%
  Batch 801: Loss = 0.5273, Accuracy = 79.69%
  Batch 901: Loss = 0.3519, Accuracy = 87.50%
=